In [ ]:
!pip -q install torch torchvision pillow pandas tqdm jiwer evaluate
!pip -q install "transformers>=4.45.0" accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import glob
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import evaluate
from jiwer import wer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_FOLDER = "/content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset"
TRAIN_FOLDER = BASE_FOLDER + "/Training"
VAL_FOLDER = BASE_FOLDER + "/Validation"
TEST_FOLDER = BASE_FOLDER + "/Testing"

In [ ]:
df_train = pd.read_csv(TRAIN_FOLDER + "/training_labels.csv")
df_train.drop(columns=["GENERIC_NAME"], inplace=True)
df_train

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
3115,3115.png,Zithrin
3116,3116.png,Zithrin
3117,3117.png,Zithrin
3118,3118.png,Zithrin


In [ ]:
df_val = pd.read_csv(VAL_FOLDER + "/validation_labels.csv")
df_val.drop(columns=["GENERIC_NAME"], inplace=True)
df_val

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
775,775.png,Zithrin
776,776.png,Zithrin
777,777.png,Zithrin
778,778.png,Zithrin


In [ ]:
df_test = pd.read_csv(TEST_FOLDER + "/testing_labels.csv")
df_test.drop(columns=["GENERIC_NAME"], inplace=True)
df_test

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
775,775.png,Zithrin
776,776.png,Zithrin
777,777.png,Zithrin
778,778.png,Zithrin


In [ ]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1
MAX_NEW_TOKENS = 128


In [ ]:
def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

cer_metric = evaluate.load("cer")

def qwen2vl_ocr_one(pil_img: Image.Image) -> str:
    """Run OCR with Qwen2-VL by prompting it to extract text."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_img},
                {"type": "text", "text": "Extract ALL text from this image exactly as it appears. Preserve line breaks."}
            ],
        }
    ]

    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    # remove prompt tokens
    gen_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    out = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return out.strip()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [ ]:
from PIL import Image
import os

TRAIN_IMG_DIR = os.path.join(TRAIN_FOLDER, "training_words")
VAL_IMG_DIR   = os.path.join(VAL_FOLDER,   "validation_words")
TEST_IMG_DIR  = os.path.join(TEST_FOLDER,  "testing_words")

SPLIT_TO_DIR = {
    "train": TRAIN_IMG_DIR,
    "val":   VAL_IMG_DIR,
    "test":  TEST_IMG_DIR,
}

def load_rgb(img_name: str, split_name: str):
    img_path = os.path.join(SPLIT_TO_DIR[split_name], img_name)
    return Image.open(img_path).convert("RGB")


def evaluate_split(df: pd.DataFrame, split_name: str, limit: int | None = None):
    if len(df) == 0:
        return None, None

    if limit is not None:
        df = df.iloc[:limit].copy()

    preds_raw = []
    gts_raw = df["MEDICINE_NAME"].astype(str).tolist()

    for start in tqdm(range(0, len(df), BATCH_SIZE), desc=f"Qwen2-VL {split_name}"):
        batch = df.iloc[start:start+BATCH_SIZE]
        for img_name in batch["IMAGE"].tolist():
            img = load_rgb(img_name, split_name)
            preds_raw.append(qwen2vl_ocr_one(img))

    out = df.copy()
    out["split"] = split_name
    out["pred_raw"] = preds_raw
    out["pred_norm"] = out["pred_raw"].apply(normalize_text)
    out["gt_norm"] = out["MEDICINE_NAME"].apply(normalize_text)

    exact_match = (out["pred_norm"] == out["gt_norm"]).mean()
    cer_value = cer_metric.compute(predictions=out["pred_norm"].tolist(),
                                   references=out["gt_norm"].tolist())
    wer_value = wer(out["gt_norm"].tolist(), out["pred_norm"].tolist())

    metrics = {
        "split": split_name,
        "samples": len(out),
        "exact_match": float(exact_match),
        "cer": float(cer_value),
        "wer": float(wer_value),
    }
    return metrics, out


LIMIT = None

all_metrics = []
all_outputs = []

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    metrics, out = evaluate_split(df, split_name, limit=LIMIT)
    if metrics is None:
        print(f"[WARN] No samples found for split: {split_name}")
        continue
    all_metrics.append(metrics)
    all_outputs.append(out)

metrics_df = pd.DataFrame(all_metrics)
print("\n===== SUMMARY =====")
print(metrics_df.to_string(index=False))


for out in all_outputs:
    split = out["split"].iloc[0]
    out_csv = os.path.join(BASE_FOLDER, f"qwen2vl_predictions_{split}.csv")
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

combined = pd.concat(all_outputs, ignore_index=True)
combined_csv = os.path.join(BASE_FOLDER, "qwen2vl_predictions_all_splits.csv")
combined.to_csv(combined_csv, index=False)
print("Saved:", combined_csv)

metrics_csv = os.path.join(BASE_FOLDER, "qwen2vl_metrics_summary.csv")
metrics_df.to_csv(metrics_csv, index=False)
print("Saved:", metrics_csv)



Qwen2-VL test: 100%|██████████| 780/780 [09:18<00:00,  1.40it/s]


===== SUMMARY =====
split  samples  exact_match      cer      wer
train     3120     0.261538 0.532114 0.792722
  val      780     0.285897 0.604268 0.803797
 test      780     0.270513 0.493699 0.762025
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_train.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_val.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_test.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_all_splits.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

def ensure_cols(df):
    df = df.copy()
    df["IMAGE"] = df["IMAGE"].astype(str)
    df["MEDICINE_NAME"] = df["MEDICINE_NAME"].astype(str)
    return df

def load_existing_preds(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return None

def evaluate_split_resumable(
    df: pd.DataFrame,
    split_name: str,
    limit: int | None = None,
    chunk_size: int = 300,
    save_every: int = 50
):
    """
    Writes/updates:
      BASE_FOLDER/qwen2vl_predictions_{split}_partial.csv
    Returns:
      path to partial csv
    """
    df = ensure_cols(df)
    if limit is not None:
        df = df.iloc[:limit].copy()

    partial_csv = os.path.join(BASE_FOLDER, f"qwen2vl_predictions_{split_name}_partial.csv")

    existing = load_existing_preds(partial_csv)
    done_images = set()
    if existing is not None and "IMAGE" in existing.columns:
        done_images = set(existing["IMAGE"].astype(str).tolist())
        print(f"✅ Resuming {split_name}: found {len(done_images)} already predicted rows.")


    remaining = df[~df["IMAGE"].isin(done_images)].copy()
    if len(remaining) == 0:
        print(f"✅ {split_name}: nothing left to predict.")
        return partial_csv

    rows_to_append = []
    processed = 0

    for chunk_start in range(0, len(remaining), chunk_size):
        chunk = remaining.iloc[chunk_start:chunk_start + chunk_size].copy()

        print(f"\n--- {split_name} chunk {chunk_start} → {chunk_start + len(chunk)} (of {len(remaining)}) ---")

        for idx, row in tqdm(chunk.iterrows(), total=len(chunk), desc=f"Qwen2-VL {split_name}"):
            img_name = row["IMAGE"]
            gt = row["MEDICINE_NAME"]

            img = load_rgb(img_name, split_name)
            pred_raw = qwen2vl_ocr_one(img)

            rec = {
                "split": split_name,
                "IMAGE": img_name,
                "MEDICINE_NAME": gt,
                "pred_raw": pred_raw,
                "pred_norm": normalize_text(pred_raw),
                "gt_norm": normalize_text(gt),
            }
            rows_to_append.append(rec)
            processed += 1


            if processed % save_every == 0:
                append_df = pd.DataFrame(rows_to_append)
                if os.path.exists(partial_csv):
                    append_df.to_csv(partial_csv, mode="a", header=False, index=False)
                else:
                    append_df.to_csv(partial_csv, index=False)
                rows_to_append = []
                print(f"💾 Saved progress: {processed} new rows -> {partial_csv}")


        if rows_to_append:
            append_df = pd.DataFrame(rows_to_append)
            if os.path.exists(partial_csv):
                append_df.to_csv(partial_csv, mode="a", header=False, index=False)
            else:
                append_df.to_csv(partial_csv, index=False)
            rows_to_append = []
            print(f"💾 Saved chunk -> {partial_csv}")

    print(f"✅ Done predicting split {split_name}. Output: {partial_csv}")
    return partial_csv

In [ ]:
LIMIT = None
CHUNK_SIZE = 300
SAVE_EVERY = 25

train_csv = evaluate_split_resumable(df_train, "train", limit=LIMIT, chunk_size=CHUNK_SIZE, save_every=SAVE_EVERY)
val_csv   = evaluate_split_resumable(df_val,   "val",   limit=LIMIT, chunk_size=CHUNK_SIZE, save_every=SAVE_EVERY)
test_csv  = evaluate_split_resumable(df_test,  "test",  limit=LIMIT, chunk_size=CHUNK_SIZE, save_every=SAVE_EVERY)


--- train chunk 0 → 300 (of 3120) ---


Qwen2-VL train:   8%|▊         | 25/300 [00:10<01:46,  2.58it/s]

💾 Saved progress: 25 new rows -> /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_train_partial.csv


Qwen2-VL train:  17%|█▋        | 50/300 [00:24<02:14,  1.86it/s]

💾 Saved progress: 50 new rows -> /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_train_partial.csv


...

💾 Saved progress: 750 new rows -> /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_test_partial.csv


Qwen2-VL test:  97%|█████████▋| 175/180 [00:58<00:01,  3.36it/s]

💾 Saved progress: 775 new rows -> /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_test_partial.csv


Qwen2-VL test: 100%|██████████| 180/180 [00:59<00:00,  3.01it/s]

💾 Saved chunk -> /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_test_partial.csv
✅ Done predicting split test. Output: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_test_partial.csv


In [ ]:
import os
import pandas as pd
from jiwer import wer

def compute_metrics_from_csv(csv_path: str, split_name: str):
    if not csv_path or not os.path.exists(csv_path):
        print(f"[WARN] Missing file for {split_name}: {csv_path}")
        return None, None

    out = pd.read_csv(csv_path)

    out["IMAGE"] = out["IMAGE"].astype(str)
    out = out.drop_duplicates(subset=["IMAGE"], keep="last")

    exact_match = (out["pred_norm"] == out["gt_norm"]).mean()
    cer_value = cer_metric.compute(
        predictions=out["pred_norm"].astype(str).tolist(),
        references=out["gt_norm"].astype(str).tolist()
    )
    wer_value = wer(out["gt_norm"].astype(str).tolist(), out["pred_norm"].astype(str).tolist())

    metrics = {
        "split": split_name,
        "samples": int(len(out)),
        "exact_match": float(exact_match),
        "cer": float(cer_value),
        "wer": float(wer_value),
    }
    return metrics, out

all_metrics = []
all_outputs = []

for split_name in ["train", "val", "test"]:
    csv_path = os.path.join(BASE_FOLDER, f"qwen2vl_predictions_{split_name}_partial.csv")
    metrics, out = compute_metrics_from_csv(csv_path, split_name)
    if metrics is None:
        continue
    all_metrics.append(metrics)
    all_outputs.append(out)

metrics_df = pd.DataFrame(all_metrics)
print("\n===== FINAL SUMMARY =====")
print(metrics_df.to_string(index=False))


for out in all_outputs:
    split = out["split"].iloc[0]
    final_csv = os.path.join(BASE_FOLDER, f"qwen2vl_predictions_{split}.csv")
    out.to_csv(final_csv, index=False)
    print("Saved:", final_csv)

combined = pd.concat(all_outputs, ignore_index=True)
combined_csv = os.path.join(BASE_FOLDER, "qwen2vl_predictions_all_splits.csv")
combined.to_csv(combined_csv, index=False)
print("Saved:", combined_csv)

metrics_csv = os.path.join(BASE_FOLDER, "qwen2vl_metrics_summary.csv")
metrics_df.to_csv(metrics_csv, index=False)
print("Saved:", metrics_csv)


===== FINAL SUMMARY =====
split  samples  exact_match      cer      wer
train     3120     0.261538 0.532114 0.792722
  val      780     0.285897 0.604268 0.803797
 test      780     0.270513 0.493699 0.762025
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_train.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_val.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_test.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/qwen2vl_predictions_all_splits.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescr